In [ ]:
import gc
import time
import json
import os
from datetime import timedelta, datetime
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython import display
import seaborn as sns
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, lr_scheduler
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import k_hop_subgraph
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import checkpoint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, precision_recall_curve, auc, confusion_matrix
from safetensors.torch import save_model as save_safetensors, load_model as load_safetensors

from rdkit import Chem,DataStructs
from rdkit.Chem import rdFingerprintGenerator, Descriptors, MACCSkeys, AllChem


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"DEVICE: {DEVICE}")
DEVICE_TYPE = DEVICE.type
if DEVICE_TYPE == 'cuda':
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    gc.collect()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(" Memory optimization enabled")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    print(f"- GPU Memory Freed. Allocated: {torch.cuda.memory_allocated()/1024**2:.2f} MB")
    print(f"- Reserved: {torch.cuda.memory_reserved()/1024**2:.2f} MB")
else:
    print("-Running on CPU (No CUDA clear needed)")

TIMESTAMP = datetime.now().strftime("%d_%b_%H-%M")

required_directories = ['images', 'models']
for folder in required_directories:
    if not os.path.exists(folder):
        print(f"✘ Directory `{folder}/` not found \nCreating...")
        os.makedirs(folder)
    else:
        print(f"✓ Directory `{folder}/` exists")


In [ ]:
DDIS_DATA_PATH = 'dataset/drugdata/ddis.csv'
DRUG_SMILE_DATA_PATH = 'dataset/drugdata/drug_smiles.csv'
DDI_TYPE_MAP = 'dataset/drugdata/ddi_type_mapping.json'
MODEL_WEIGHT_PATH = f"models/PharmGAT_v2_{TIMESTAMP}"
MODEL_METADATA_PATH = f"models/PharmGAT_v2-metadata_{TIMESTAMP}"


In [ ]:
CONFIG = {
    "physical_batch_size": 512,
    "accumulation_steps": 4,
    "hidden_dim": 384,           
    "n_heads": 6,               
    "dropout": 0.15,         
    "n_gat_layers":3,           

    "lr": 3e-4,
    "weight_decay": 5e-5,
    "epochs": 250,

    "type_loss_weight": 0.3,
    "use_focal_loss": True,
    "use_class_weights": False,
    "focal_alpha": 0.5,
    "focal_gamma": 2.0,

    "scheduler_patience": 7,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,

    "full_graph_on_gpu": True,
    "validate_every_n_epochs": 3,
    "device_platform": "Local",

    "use_data_aug": True,
    "edge_dropout_rate": 0.05,
    "feature_noise": 0.005,
}


In [ ]:
def read_data(ddi_path: str = DDIS_DATA_PATH, smiles_path: str = DRUG_SMILE_DATA_PATH):
    ddi_df = pd.read_csv(ddi_path)
    smiles_df = pd.read_csv(smiles_path)
    print("Data loaded successfully")
    print("DDI shape:", ddi_df.shape)
    unique_drugs_ddi = set(ddi_df['d1'].unique()) | set(ddi_df['d2'].unique())
    print(f"Total DDI pairs: {len(ddi_df)}")
    print(f"Unique drugs in DDI: {len(unique_drugs_ddi)}")
    print(f"Drugs with SMILES: {len(smiles_df)}")
    print(f"Drug Pair Interaction type 0: {(ddi_df['type'] == 0).sum()}")
    print(f"Drug Pair Interaction type 7: {(ddi_df['type'] == 7).sum()}")
    print("SMILES shape:", smiles_df.shape)
    print("DDI columns:", ddi_df.columns.tolist())
    print("SMILES columns:", smiles_df.columns.tolist())
    drugs_with_smiles = set(smiles_df['drug_id'].unique())
    overlap = unique_drugs_ddi & drugs_with_smiles
    print(f"• Drugs with both DDI and SMILES: {len(overlap)}")
    print(f"• Coverage: {len(overlap)/len(unique_drugs_ddi)*100:.2f}%")

    return ddi_df, smiles_df

def quick_sanity_report(ddi_df: pd.DataFrame, smiles_df: pd.DataFrame):
    print("\nⓘ QUICK SANITY REPORT:\n")

    print("[1. DDI Data: Interaction pairs]")
    print("Total rows:", len(ddi_df))
    print("Nulls per column:", ddi_df.isnull().sum().to_dict())
    if {"d1", "d2", "type"}.issubset(ddi_df.columns):
        print("Total interactions types:", ddi_df["type"].nunique())
        print("Top `type` distribution (top 10):")
        print(ddi_df["type"].value_counts().head(10))

    print("\n[2. SMILES Data: Molecular Structures]")
    print("Total rows:", len(smiles_df))
    print("Checking nulls per column?:", smiles_df.isnull().sum().to_dict())

    ddi_drugs = set(ddi_df["d1"]).union(set(ddi_df["d2"]))
    smiles_drugs = set(smiles_df["drug_id"]) if "drug_id" in smiles_df.columns else set()
    overlap = ddi_drugs & smiles_drugs
    print(f"\n● Overlap: {len(overlap)}/{len(ddi_drugs)} DDI drugs have SMILES ({100*len(overlap)/len(ddi_drugs):.2f}%)")

ddi_df, smiles_df = read_data()
quick_sanity_report(ddi_df, smiles_df)


In [ ]:
class DrugFeatureExtractor:
    def get_features(self, smiles_df):
        view1_morgan = []
        view2_maccs = []
        view3_phys = []
        valid_ids = []
        print("Extracting Multi-View Features...")
        for _, row in smiles_df.iterrows():
            mol = Chem.MolFromSmiles(str(row["smiles"]))
            if mol:
                fp_bv = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
                fp = np.zeros((1024,), dtype=np.float32)
                DataStructs.ConvertToNumpyArray(fp_bv, fp)
                maccs_bv = MACCSkeys.GenMACCSKeys(mol)
                maccs = np.zeros((len(maccs_bv),), dtype=np.float32)
                DataStructs.ConvertToNumpyArray(maccs_bv, maccs)
                desc = np.array([
			Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
			Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
			Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
			Descriptors.NumAromaticRings(mol), Descriptors.FractionCSP3(mol)
                ], dtype=np.float32)
                view1_morgan.append(fp)
                view2_maccs.append(maccs)
                view3_phys.append(desc)
                valid_ids.append(row["drug_id"])
        v1 = np.stack(view1_morgan)
        v2 = np.stack(view2_maccs)
        v3 = np.stack(view3_phys)
        scaler = StandardScaler()
        v3 = scaler.fit_transform(v3)
        print(f"Processed {len(valid_ids)} drugs successfully!")
        print(f"  - View 1 Shape: {v1.shape}")
        print(f"  - View 2 Shape: {v2.shape}")
        print(f"  - View 3 Shape: {v3.shape}")
        return {
            "v1": torch.tensor(v1, dtype=torch.float),
            "v2": torch.tensor(v2, dtype=torch.float),
            "v3": torch.tensor(v3, dtype=torch.float),
            "ids": valid_ids
        }
smiles_df = pd.read_csv(DRUG_SMILE_DATA_PATH)
extractor = DrugFeatureExtractor()
features_dict = extractor.get_features(smiles_df)

In [ ]:
class GraphBuilder:
    def __init__(self, ddi_path, feature_dict):
        self.ddi_df = pd.read_csv(ddi_path)
        self.feats = feature_dict
        self.drug_map = {d: i for i, d in enumerate(feature_dict['ids'])}
        self.idx_map = {i: d for d, i in self.drug_map.items()}
        self.type_enc = LabelEncoder()

    def build(self):
        pos_src, pos_dst, pos_types = [], [], []
        neg_src, neg_dst = [], []

        print("Building DDI Graph Topology...")
        for _, row in self.ddi_df.iterrows():
            if row['d1'] in self.drug_map and row['d2'] in self.drug_map:
                u, v = self.drug_map[row['d1']], self.drug_map[row['d2']]

                pos_src.extend([u, v])
                pos_dst.extend([v, u])
                pos_types.extend([row['type'], row['type']])

                if isinstance(row['Neg samples'], str):
                    neg_raw = row['Neg samples'].strip().split('$')[0]
                    if neg_raw and neg_raw in self.drug_map:
                        w = self.drug_map[neg_raw]
                        neg_src.extend([u, w])
                        neg_dst.extend([w, u])

        pos_edge_index = torch.tensor([pos_src, pos_dst], dtype=torch.long)
        neg_edge_index = torch.tensor([neg_src, neg_dst], dtype=torch.long)

        full_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)

        y_types_enc = self.type_enc.fit_transform(pos_types)

        y_binary = torch.cat([torch.ones(pos_edge_index.shape[1]), torch.zeros(neg_edge_index.shape[1])]).long()
        y_type = torch.cat([torch.tensor(y_types_enc), torch.full((neg_edge_index.shape[1],), -1)]).long()

        return {
            "x_v1": self.feats['v1'],
            "x_v2": self.feats['v2'],
            "x_v3": self.feats['v3'],
            "edge_index": full_edge_index,
            "y_binary": y_binary,
            "y_type": y_type,
            "n_types": len(self.type_enc.classes_),
            "encoder": self.type_enc,
            "drug_map": self.drug_map
        }

    def create_stratified_split(self, graph):
        print("Creating Stratified Splits (TDCommons Standard)...")
        n_edges = graph['edge_index'].shape[1]
        indices = np.arange(n_edges)

        train_idx, temp_idx = train_test_split(
            indices, test_size=0.3, stratify=graph['y_binary'], random_state=42
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, stratify=graph['y_binary'][temp_idx], random_state=42
        )

        for name, idx in zip(['train', 'val', 'test'], [train_idx, val_idx, test_idx]):
            mask = torch.zeros(n_edges, dtype=torch.bool)
            mask[idx] = True
            graph[f'{name}_mask'] = mask

        print(f"  - Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
        return graph

builder = GraphBuilder(DDIS_DATA_PATH, features_dict)
graph_data = builder.build()
graph = builder.create_stratified_split(graph_data)


In [ ]:
class DDIDataAugmenter:
    @staticmethod
    def augment_graph_training(graph, edge_dropout=0.1, feature_noise=0.01):
        if edge_dropout > 0:
            mask = torch.rand(graph['edge_index'].shape[1]) > edge_dropout
            aug_edge_index = graph['edge_index'][:, mask]
        else:
            aug_edge_index = graph['edge_index']
        if feature_noise > 0:
            aug_x_v1 = graph['x_v1'] + torch.randn_like(graph['x_v1']) * feature_noise
            aug_x_v2 = graph['x_v2'] + torch.randn_like(graph['x_v2']) * feature_noise
            aug_x_v3 = graph['x_v3'] + torch.randn_like(graph['x_v3']) * feature_noise
        else:
            aug_x_v1, aug_x_v2, aug_x_v3 = graph['x_v1'], graph['x_v2'], graph['x_v3']

        return {
            'edge_index': aug_edge_index,
            'x_v1': aug_x_v1,
            'x_v2': aug_x_v2,
            'x_v3': aug_x_v3
        }


In [ ]:
class FeatureAttentionLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.atten = nn.Linear(dim, 1)

    def forward(self, v1, v2, v3):
        stack = torch.stack([v1, v2, v3], dim=1)
        scores = F.softmax(self.atten(stack), dim=1)
        fused = torch.sum(stack * scores, dim=1)
        return fused, scores

class GATv2NN(nn.Module):
    def __init__(self, dims, hidden_dim, n_heads, n_types, dropout=0.3):
        super().__init__()

        self.proj_v1 = nn.Sequential(
            nn.Linear(dims['v1'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        self.proj_v2 = nn.Sequential(
            nn.Linear(dims['v2'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        self.proj_v3 = nn.Sequential(
            nn.Linear(dims['v3'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )

        self.feat_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )

        self.gat1 = GATv2Conv(
            hidden_dim, hidden_dim // n_heads, 
            heads=n_heads, concat=True, dropout=dropout
        )
        self.norm1 = nn.LayerNorm(hidden_dim)

        self.gat2 = GATv2Conv(
            hidden_dim, hidden_dim, 
            heads=1, concat=False, dropout=dropout
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.gat3  = GATv2Conv(
            hidden_dim, hidden_dim,
            heads=1, concat=False, dropout=dropout
        ) 
        self.norm3 = nn.LayerNorm(hidden_dim)

        self.edge_encoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )

        self.head_bin = nn.Linear(hidden_dim // 2, 2)
        self.head_type = nn.Linear(hidden_dim // 2, n_types)

        self.dropout = dropout

        print("Model Intialized\n")

    def get_node_embeddings(self, x_v1, x_v2, x_v3, edge_index):
        h1 = self.proj_v1(x_v1)
        h2 = self.proj_v2(x_v2)
        h3 = self.proj_v3(x_v3)

        stack = torch.stack([h1, h2, h3], dim=1)
        scores = F.softmax(self.feat_attention(stack), dim=1)
        h_fused = torch.sum(stack * scores, dim=1)

        if self.training:
            h_gat1 = checkpoint(self.gat1, h_fused, edge_index, use_reentrant=False)
            h_gat1 = self.norm1(h_gat1 + h_fused)
            h_gat1 = F.elu(h_gat1)

            h_gat2 = checkpoint(self.gat2, h_gat1, edge_index, use_reentrant=False)
            node_emb = self.norm2(h_gat2 + h_fused)

            h_gat3 = checkpoint(self.gat3, node_emb, edge_index, use_reentrant=False)
            node_emb = self.norm3(h_gat3 + h_fused)
        else:
            h_gat1 = self.norm1(self.gat1(h_fused, edge_index) + h_fused)
            h_gat1 = F.elu(h_gat1)
            h_gat2 = self.gat2(h_gat1, edge_index)
            node_emb = self.norm2(h_gat2 + h_fused)
            h_gat3 = self.gat3(node_emb, edge_index)
            node_emb = self.norm3(h_gat3 + h_fused)

        return node_emb

    def forward_edges_from_emb(self, node_emb, edge_label_index):
        src, dst = edge_label_index[0], edge_label_index[1]
        edge_feat = torch.cat([node_emb[src], node_emb[dst]], dim=-1)
        shared = self.edge_encoder(edge_feat)
        return self.head_bin(shared), self.head_type(shared)

    def forward(self, x_v1, x_v2, x_v3, edge_index, edge_label_index):
        node_emb = self.get_node_embeddings(x_v1, x_v2, x_v3, edge_index)
        return self.forward_edges_from_emb(node_emb, edge_label_index) 


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.7, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()


In [ ]:
class BestModelTracker:
    def __init__(self, save_dir='models', min_delta=0.005, metric_name='f1'):
        self.save_dir = save_dir
        self.min_delta = min_delta     
        self.metric_name = metric_name
        self.best_score = 0.0
        self.best_path = None
        self.save_count = 0
        os.makedirs(save_dir, exist_ok=True)

    def update(self, model, score, epoch):
        improvement = score - self.best_score

        if improvement > self.min_delta:
            if self.best_path and os.path.exists(self.best_path):
                os.remove(self.best_path)
            self.best_score = score
            filename = f'PharmGAT_v2_ep-{epoch}_{TIMESTAMP}.safetensors'
            self.best_path = os.path.join(self.save_dir, filename)

            base_model = model._orig_mod if hasattr(model, '_orig_mod') else model
            save_safetensors(base_model, self.best_path)

            self.save_count += 1
            print(f"New best model found! At epoch {epoch} with F1: {score:.4f} and improvement of {improvement:.4f}")
            return True

        return False

    def load_best(self, model):
        if self.best_path and os.path.exists(self.best_path):
            base_model = model._orig_mod if hasattr(model, '_orig_mod') else model
            load_safetensors(base_model, self.best_path)
            print(f"(✓) Loaded best model: {os.path.basename(self.best_path)}")
            print(f"   Best {self.metric_name}: {self.best_score:.4f}")
        else:
            print("(✘) No saved model found.")
        return model

    def summary(self):
        print(f"\n➜ Model Saving Summary:")
        print(f"- Total progressive best models: {self.save_count}")
        print(f"- Best {self.metric_name}: {self.best_score:.4f}")
        print(f"- Saved at: {self.best_path}")


In [ ]:
def train_model(graph, config, model, device='cuda'):
    physical_batch = config["physical_batch_size"]
    accum_steps = config["accumulation_steps"]
    type_w = config.get("type_loss_weight", 0.5)
    validate_every = config.get("validate_every_n_epochs", 2)

    START_TIME = time.perf_counter()
    print(f"MODEL TRAINING STARTING AT: {time.strftime('%H:%M:%S', time.localtime())}...\n")

    train_edges = graph['edge_index'][:, graph['train_mask']]
    train_labels = torch.stack([
        graph['y_binary'][graph['train_mask']],
        graph['y_type'][graph['train_mask']]
    ], dim=1)
    val_edges = graph['edge_index'][:, graph['val_mask']]
    val_labels = torch.stack([
        graph['y_binary'][graph['val_mask']],
        graph['y_type'][graph['val_mask']]
    ], dim=1)

    train_loader = DataLoader(
        TensorDataset(train_edges.t(), train_labels),
        batch_size=physical_batch, shuffle=True,
        pin_memory=True, num_workers=2, persistent_workers=True
    )
    val_loader = DataLoader(
        TensorDataset(val_edges.t(), val_labels),
        batch_size=physical_batch * 2, shuffle=False,
        pin_memory=True, num_workers=2
    )

    y_train_bin = graph['y_binary'][graph['train_mask']].numpy()
    n_pos = int((y_train_bin == 1).sum())
    n_neg = int((y_train_bin == 0).sum())

    use_focal = config.get("use_focal_loss", False)
    use_weights = config.get("use_class_weights", False)

    if use_focal and use_weights:
        print("⚠︎ Warning: Both focal loss and class weights enabled. Using Focal Loss (takes priority).")
        use_weights = False

    if use_focal:
        crit_bin = FocalLoss(
            alpha=config.get("focal_alpha", 0.7),
            gamma=config.get("focal_gamma", 2.0)
        )
        print(f"✓ Loss: FocalLoss(alpha={config.get('focal_alpha', 0.7)}, gamma={config.get('focal_gamma', 2.0)})")

    elif use_weights and n_pos > 0 and n_neg > 0:
        w_pos = (n_pos + n_neg) / (2.0 * n_pos)
        w_neg = (n_pos + n_neg) / (2.0 * n_neg)
        class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32, device=device)
        crit_bin = nn.CrossEntropyLoss(weight=class_weights)
        print(f"✓ Loss: CrossEntropyLoss | Class weights: neg={w_neg:.3f}, pos={w_pos:.3f}")

    else:
        crit_bin = nn.CrossEntropyLoss()
        print("✓ Loss: CrossEntropyLoss (unweighted)")

    crit_type = nn.CrossEntropyLoss(ignore_index=-1)

    full_adj = graph['edge_index'].to(device)
    x_v1 = graph['x_v1'].to(device)
    x_v2 = graph['x_v2'].to(device)
    x_v3 = graph['x_v3'].to(device)

    gnn_param_names = ['proj_v1', 'proj_v2', 'proj_v3', 'feat_attention', 
                   'gat1', 'gat2', 'gat3', 'norm1', 'norm2', 'norm3']
    edge_param_names = ['edge_encoder', 'head_bin', 'head_type']

    base_model = model._orig_mod if hasattr(model, '_orig_mod') else model

    gnn_params = [p for n, p in base_model.named_parameters()
                  if any(n.startswith(x) for x in gnn_param_names)]
    edge_params = [p for n, p in base_model.named_parameters()
                   if any(n.startswith(x) for x in edge_param_names)]

    print(f"- GNN params:  {sum(p.numel() for p in gnn_params):,}")
    print(f"- Edge params: {sum(p.numel() for p in edge_params):,}")

    gnn_optimizer  = Adam(gnn_params,  lr=config['lr'], weight_decay=config.get('weight_decay', 1e-4))
    edge_optimizer = Adam(edge_params, lr=config['lr'], weight_decay=config.get('weight_decay', 1e-4))

    edge_scaler = GradScaler()

    gnn_scheduler = lr_scheduler.ReduceLROnPlateau(
        gnn_optimizer, mode='max',
        patience=config.get('scheduler_patience', 4),
        factor=config.get('scheduler_factor', 0.5)
    )
    edge_scheduler = lr_scheduler.ReduceLROnPlateau(
        edge_optimizer, mode='max',
        patience=config.get('scheduler_patience', 4),
        factor=config.get('scheduler_factor', 0.5)
    )

    history = {
        'train_loss': [], 'val_acc': [], 'val_f1': [],
        'val_precision': [], 'val_recall': [], 'training_duration': []
    }

    print(f"\n{'—'*75}")
    print(f"{'Epoch':<6} | {'Train Loss':<11} | {'Val Acc':<8} | {'Val F1':<8} | {'Val P':<8} | {'Val R':<8} | {'Time':<8}")
    print(f"{'—'*75}")

    best_f1 = 0.0

    tracker = BestModelTracker(save_dir='models',min_delta=0.005, metric_name='f1')

    for epoch in range(1, config['epochs'] + 1):
        epoch_start = time.time()
        model.train()

        use_aug = config.get("use_data_aug", True)

        if use_aug and (config.get('edge_dropout_rate', 0) > 0 or config.get('feature_noise', 0) > 0):
            aug_data = DDIDataAugmenter.augment_graph_training(
                {'edge_index': full_adj, 'x_v1': x_v1, 'x_v2': x_v2, 'x_v3': x_v3},
                edge_dropout=config.get('edge_dropout_rate', 0),
                feature_noise=config.get('feature_noise', 0)
            )
            aug_adj = aug_data['edge_index']
            aug_x_v1 = aug_data['x_v1']
            aug_x_v2 = aug_data['x_v2']
            aug_x_v3 = aug_data['x_v3']

        else:
            aug_adj = full_adj
            aug_x_v1, aug_x_v2, aug_x_v3 = x_v1, x_v2, x_v3

        gnn_optimizer.zero_grad()
        with autocast(device.type):            
            node_emb = model.get_node_embeddings(aug_x_v1, aug_x_v2, aug_x_v3, aug_adj)

        node_emb_proxy = node_emb.detach().requires_grad_(True) 

        edge_optimizer.zero_grad()
        total_loss = 0.0       

        for i, (batch_edges, batch_labels) in enumerate(train_loader):
            batch_edges = batch_edges.t().to(device, non_blocking=True)
            y_bin  = batch_labels[:, 0].to(device, non_blocking=True)
            y_type = batch_labels[:, 1].to(device, non_blocking=True)

            with autocast(device.type):
                pred_bin, pred_type = model.forward_edges_from_emb(node_emb_proxy, batch_edges)
                loss = (crit_bin(pred_bin, y_bin) +
                        type_w * crit_type(pred_type, y_type)) / accum_steps

            edge_scaler.scale(loss).backward()
            total_loss += loss.item() * accum_steps

            if (i + 1) % accum_steps == 0:
                edge_scaler.step(edge_optimizer)
                edge_scaler.update()
                edge_optimizer.zero_grad()

        train_loss = total_loss / len(train_loader)
        history['train_loss'].append(train_loss)

        if node_emb_proxy.grad is not None:
            scale = edge_scaler.get_scale()
            unscaled_grad = node_emb_proxy.grad / scale if scale != 0 else node_emb_proxy.grad
            node_emb.backward(unscaled_grad)
            torch.nn.utils.clip_grad_norm_(gnn_params, max_norm=1.0)
            gnn_optimizer.step()
        gnn_optimizer.zero_grad()

        if use_aug:
            del aug_adj, aug_x_v1, aug_x_v2, aug_x_v3

        do_validate = (epoch % validate_every == 0) or (epoch == config['epochs'])

        if do_validate:
            model.eval()
            all_preds, all_trues = [], []

            with torch.no_grad():
                with autocast(device.type):
                    node_emb_val = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
                for batch_edges, batch_labels in val_loader:
                    batch_edges = batch_edges.t().to(device, non_blocking=True)
                    p_bin, _ = model.forward_edges_from_emb(node_emb_val, batch_edges)
                    all_preds.extend(torch.argmax(p_bin, dim=1).cpu().numpy())
                    all_trues.extend(batch_labels[:, 0].numpy())

            all_trues = np.array(all_trues)
            all_preds = np.array(all_preds)

            val_acc  = accuracy_score(all_trues, all_preds)
            val_f1   = f1_score(all_trues, all_preds, zero_division=0)
            val_prec = precision_score(all_trues, all_preds, zero_division=0)
            val_rec  = recall_score(all_trues, all_preds, zero_division=0)

            history['val_acc'].append(val_acc)
            history['val_f1'].append(val_f1)
            history['val_precision'].append(val_prec)
            history['val_recall'].append(val_rec)

            gnn_scheduler.step(val_f1)
            edge_scheduler.step(val_f1)

            tracker.update(model, val_f1, epoch)
            if val_f1 > best_f1:
                best_f1 = val_f1

            epoch_time = time.time() - epoch_start
            print(f"{epoch:>4}   | {train_loss:>9.4f}   | {val_acc:>6.4f}  | {val_f1:>6.4f}  | {val_prec:>6.4f}  | {val_rec:>6.4f}  | {epoch_time:>5.1f}s")
        else:
            for k in ['val_acc', 'val_f1', 'val_precision', 'val_recall']:
                history[k].append(history[k][-1] if history[k] else 0.0)

        torch.cuda.empty_cache()

    tracker.summary()
    model = tracker.load_best(model)

    END_TIME = time.perf_counter()
    formatted_time = str(timedelta(seconds=int(END_TIME - START_TIME)))
    history['training_duration'].append(formatted_time)

    print(f"{'—'*75}")
    print(f"{config['epochs']} Epochs completed in {history['training_duration']} "
          f"at {time.strftime('%H:%M:%S', time.localtime())} with best Val F1: {best_f1:.4f}")

    return model, history, tracker


In [ ]:
model = GATv2NN(
    dims={'v1': 1024, 'v2': 167, 'v3': 8},
    hidden_dim=CONFIG['hidden_dim'],
    n_heads=CONFIG['n_heads'],
    n_types=graph['n_types'],
    dropout=CONFIG['dropout']
).to(DEVICE)

trained_model, training_history, tracker = train_model(
    graph, 
    CONFIG, 
    model, 
    device=DEVICE
)


In [ ]:
def plot_training_dashboard(history, config):
    display.clear_output(wait=True)

    total_epochs = len(history['train_loss'])
    validate_every = config.get('validate_every_n_epochs', 1)

    validated_epochs = []
    for epoch in range(1, total_epochs + 1):
        do_validate = (epoch % validate_every == 0) or (epoch == total_epochs)
        if do_validate:
            validated_epochs.append(epoch)

    validated_indices = [e - 1 for e in validated_epochs]
    train_loss_validated = [history['train_loss'][i] for i in validated_indices]
    val_acc_validated = [history['val_acc'][i] for i in validated_indices]
    val_f1_validated = [history['val_f1'][i] for i in validated_indices]
    val_precision_validated = [history['val_precision'][i] for i in validated_indices]
    val_recall_validated = [history['val_recall'][i] for i in validated_indices]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"Training Metrics Dashboard (Validated Epochs Only)\nValidation Frequency: Every {validate_every} epoch(s)", 
                 fontsize=16, fontweight='bold')
    axes[0, 0].plot(validated_epochs, train_loss_validated, 
                    color='royalblue', marker='o', markersize=5, linewidth=2, label='Train Loss')
    axes[0, 0].set_title("Training Loss", fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].grid(alpha=0.3, linestyle='--')
    axes[0, 0].legend()

    if len(train_loss_validated) > 0:
        axes[0, 0].annotate(f'{train_loss_validated[-1]:.4f}', 
                           xy=(validated_epochs[-1], train_loss_validated[-1]),
                           xytext=(5, 5), textcoords='offset points', fontsize=9)

    axes[0, 1].plot(validated_epochs, val_acc_validated, 
                    color='seagreen', marker='o', markersize=5, linewidth=2, label='Val Accuracy')
    axes[0, 1].set_title("Validation Accuracy", fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].grid(alpha=0.3, linestyle='--')
    axes[0, 1].legend()

    if len(val_acc_validated) > 0:
        best_acc_idx = val_acc_validated.index(max(val_acc_validated))
        axes[0, 1].scatter([validated_epochs[best_acc_idx]], [val_acc_validated[best_acc_idx]], 
                          color='red', s=100, zorder=5, marker='*', label='Best')
        axes[0, 1].annotate(f'Best: {val_acc_validated[best_acc_idx]:.4f}', 
                           xy=(validated_epochs[best_acc_idx], val_acc_validated[best_acc_idx]),
                           xytext=(5, -15), textcoords='offset points', fontsize=9, color='red')
        axes[0, 1].legend()

    axes[1, 0].plot(validated_epochs, val_f1_validated, 
                    color='darkorange', marker='o', markersize=5, linewidth=2, label='Val F1')
    axes[1, 0].set_title("Validation F1-Score", fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("F1-Score")
    axes[1, 0].grid(alpha=0.3, linestyle='--')
    axes[1, 0].legend()

    if len(val_f1_validated) > 0:
        best_f1_idx = val_f1_validated.index(max(val_f1_validated))
        axes[1, 0].scatter([validated_epochs[best_f1_idx]], [val_f1_validated[best_f1_idx]], 
                          color='red', s=100, zorder=5, marker='*', label='Best')
        axes[1, 0].annotate(f'Best: {val_f1_validated[best_f1_idx]:.4f}', 
                           xy=(validated_epochs[best_f1_idx], val_f1_validated[best_f1_idx]),
                           xytext=(5, -15), textcoords='offset points', fontsize=9, color='red')
        axes[1, 0].legend()

    axes[1, 1].plot(validated_epochs, val_precision_validated, 
                    color='crimson', marker='o', markersize=5, linewidth=2, label='Precision')
    axes[1, 1].plot(validated_epochs, val_recall_validated, 
                    color='purple', marker='s', markersize=5, linewidth=2, label='Recall')
    axes[1, 1].set_title("Validation Precision & Recall", fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Score")
    axes[1, 1].grid(alpha=0.3, linestyle='--')
    axes[1, 1].legend()

    plt.tight_layout()
    display.display(plt.gcf())    
    plt.savefig(f"images/AushadiNet_training_dashboard{TIMESTAMP}.png", 
                dpi=150, bbox_inches='tight')    
    plt.close(fig)

plot_training_dashboard(training_history, CONFIG)


In [ ]:
def visualize_drug_interaction_graph(graph, num_nodes=150, seed=42):
    print("● Generating Graph Network Visualization...")

    edge_index = graph['edge_index'].cpu().numpy()
    y_binary = graph['y_binary'].cpu().numpy()
    total_nodes = graph['x_v1'].shape[0]

    print(f"  - Total nodes in graph: {total_nodes}")
    print(f"  - Total edges in graph: {edge_index.shape[1]}")

    degree_count = {}
    for i in range(edge_index.shape[1]):
        src, dst = int(edge_index[0, i]), int(edge_index[1, i])
        degree_count[src] = degree_count.get(src, 0) + 1
        degree_count[dst] = degree_count.get(dst, 0) + 1

    sorted_nodes = sorted(degree_count.items(), key=lambda x: x[1], reverse=True)
    selected_nodes = [node for node, deg in sorted_nodes[:num_nodes]]
    selected_nodes_set = set(selected_nodes)

    print(f"  - Selected {len(selected_nodes)} high-degree nodes")

    G = nx.Graph()
    G.add_nodes_from(selected_nodes)

    pos_edges = []
    neg_edges = []

    for i in range(edge_index.shape[1]):
        src, dst = int(edge_index[0, i]), int(edge_index[1, i])

        if src in selected_nodes_set and dst in selected_nodes_set and src != dst:
            edge = (src, dst)
            if y_binary[i] == 1:
                pos_edges.append(edge)
            else:
                neg_edges.append(edge)

    pos_edges = list(set(pos_edges))
    neg_edges = list(set(neg_edges))

    print(f"  - Found {len(pos_edges)} interaction edges")
    print(f"  - Found {len(neg_edges)} safe edges")

    G.add_edges_from(pos_edges)
    G.add_edges_from(neg_edges)

    isolated = list(nx.isolates(G))
    G.remove_nodes_from(isolated)
    print(f"  - Removed {len(isolated)} isolated nodes")
    print(f"  - Final graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    if G.number_of_edges() == 0:
        print("Warning: No edges found! Using alternative sampling strategy")
        G = nx.Graph()
        sample_edges = np.random.choice(edge_index.shape[1], min(1000, edge_index.shape[1]), replace=False)
        for idx in sample_edges:
            src, dst = int(edge_index[0, idx]), int(edge_index[1, idx])
            if src != dst:
                G.add_edge(src, dst, interaction=int(y_binary[idx]))

        if G.number_of_nodes() > 0:
            largest_cc = max(nx.connected_components(G), key=len)
            G = G.subgraph(largest_cc).copy()

            pos_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('interaction', 0) == 1]
            neg_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('interaction', 0) == 0]

    node_degrees = dict(G.degree())

    node_sizes = [20 + node_degrees.get(node, 0) * 8 for node in G.nodes()]

    fig, ax = plt.subplots(figsize=(24, 14), facecolor='white')

    print(" Computing graph layout...")
    pos = nx.spring_layout(G, k=2.0, iterations=60, seed=seed)

    if len(neg_edges) > 0:
        nx.draw_networkx_edges(
            G, pos,
            edgelist=neg_edges,
            width=2,
            alpha=0.7,
            edge_color='#174500',
            style='dashed',
            ax=ax
        )

    if len(pos_edges) > 0:
        nx.draw_networkx_edges(
            G, pos,
            edgelist=pos_edges,
            width=1.0,
            alpha=0.3,
            edge_color='#b52618',
            ax=ax
        )

    high_degree_nodes = [n for n in G.nodes() if node_degrees.get(n, 0) > np.percentile(list(node_degrees.values()), 75)]
    low_degree_nodes = [n for n in G.nodes() if n not in high_degree_nodes]

    if len(low_degree_nodes) > 0:
        low_colors = [node_degrees.get(node, 0) for node in low_degree_nodes]
        low_sizes = [30 + node_degrees.get(node, 0) * 8 for node in low_degree_nodes]
        low_pos = {k: v for k, v in pos.items() if k in low_degree_nodes}

        nx.draw_networkx_nodes(
            G, low_pos,
            nodelist=low_degree_nodes,
            node_size=low_sizes,
            node_color=low_colors,
            node_shape='o',
            cmap=plt.cm.viridis,
            alpha=0.9,
            edgecolors='white',
            linewidths=1.5,
            vmin=min(node_degrees.values()),
            vmax=max(node_degrees.values()),
            ax=ax
        )

    if len(high_degree_nodes) > 0:
        high_colors = [node_degrees.get(node, 0) for node in high_degree_nodes]
        high_sizes = [20 + node_degrees.get(node, 0) * 8 for node in high_degree_nodes]
        high_pos = {k: v for k, v in pos.items() if k in high_degree_nodes}

        nodes_high = nx.draw_networkx_nodes(
            G, high_pos,
            nodelist=high_degree_nodes,
            node_size=high_sizes,
            node_color=high_colors,
            node_shape='D',
            cmap=plt.cm.viridis,
            alpha=0.9,
            edgecolors='gold',
            linewidths=1.5,
            vmin=min(node_degrees.values()),
            vmax=max(node_degrees.values()),
            ax=ax
        )

        cbar = plt.colorbar(nodes_high, ax=ax, fraction=0.02, pad=0.02)
        cbar.set_label('Node Degree (Number of Connections)', fontsize=14, fontweight='bold')

    ax.set_title(
        'AushadhiNet: Drug-Drug Interaction Network Topology\n'
        f'Graph Neural Network Architecture - Message Passing Visualization',
        fontsize=22,
        fontweight='bold',
        pad=30
    )

    legend_elements = [
        plt.Line2D([0], [0], color='#E74C3C', linewidth=3,
                   label=f'DDI Edges (Adverse Interactions): {len(pos_edges)}'),
        plt.Line2D([0], [0], color='#27AE60', linewidth=2, linestyle='--',
                   label=f'Safe Edges (No Interaction): {len(neg_edges)}'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#440154',
                   markersize=14, markeredgecolor='white', markeredgewidth=1.5,
                   label='Regular Drug Nodes (circles)', linestyle='None'),
        plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='#FDE724',
                   markersize=14, markeredgecolor='gold', markeredgewidth=2,
                   label='Higher Connection (hub) Drug Nodes (diamonds)', linestyle='None')
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=13,
              framealpha=0.95, edgecolor='black', fancybox=True, shadow=True)

    avg_degree = np.mean(list(node_degrees.values())) if node_degrees else 0
    max_degree = max(node_degrees.values()) if node_degrees else 0
    density = nx.density(G)

    stats_text = (
        f"Network Statistics:\n"
        f"Nodes:          {G.number_of_nodes():>6}\n"
        f"Edges:          {G.number_of_edges():>6}\n"
        f"Interactions:   {len(pos_edges):>6}\n"
        f"Safe Pairs:     {len(neg_edges):>6}\n"
        f"Avg Degree:     {avg_degree:>6.2f}\n"
        f"Max Degree:     {max_degree:>6}\n"
        f"Density:        {density:>6.4f}\n"
        f"Hub Nodes:      {len(high_degree_nodes):>6}"
    )

    props = dict(boxstyle='round,pad=1.0', facecolor='#F8F9FA',
                 edgecolor='black', linewidth=2.5, alpha=0.95)
    ax.text(0.015, 0.015, stats_text, transform=ax.transAxes, fontsize=12,
            verticalalignment='bottom', bbox=props, family='monospace',
            fontweight='bold')

    ax.axis('off')
    plt.tight_layout()

    output_path = f"images/AushadhiNet_graph_topology_{TIMESTAMP}.png"
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\n✓ Graph visualization saved to: {output_path}")

    print(f"\n● Final Network Statistics:")
    print(f"  - Nodes (Drugs): {G.number_of_nodes()}")
    print(f"  - Edges (Total): {G.number_of_edges()}")
    print(f"  - Positive Edges (Interactions): {len(pos_edges)}")
    print(f"  - Negative Edges (Safe): {len(neg_edges)}")
    print(f"  - Hub Nodes (High Degree): {len(high_degree_nodes)}")
    print(f"  - Average Degree: {avg_degree:.2f}")
    print(f"  - Network Density: {density:.4f}")

    plt.show()
    plt.close()

    return G, pos

G, pos = visualize_drug_interaction_graph(
    graph,
    num_nodes=120,
    seed=42
)


In [ ]:
def training_evaluation(model, graph, batch_size=512):
    print("COMPREHENSIVE TEST SET EVALUATION")
    print("—"*33, "\n")

    model.eval()
    test_edges = graph['edge_index'][:, graph['test_mask']]
    test_labels_bin = graph['y_binary'][graph['test_mask']]
    test_labels_type = graph['y_type'][graph['test_mask']]

    test_loader = DataLoader(
        TensorDataset(test_edges.t(), 
                     torch.stack([test_labels_bin, test_labels_type], dim=1)),
        batch_size=batch_size, shuffle=False
    )

    full_adj = graph['edge_index'].to(DEVICE)
    x_v1, x_v2, x_v3 = graph['x_v1'].to(DEVICE), graph['x_v2'].to(DEVICE), graph['x_v3'].to(DEVICE)

    all_preds_bin, all_probs_bin, all_trues_bin = [], [], []
    all_preds_type, all_trues_type = [], []

    with torch.no_grad():
        with autocast(DEVICE_TYPE):
            node_emb = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)

        for batch_edges, batch_labels in test_loader:
            batch_edges = batch_edges.t().to(DEVICE)
            p_bin, p_type = model.forward_edges_from_emb(node_emb, batch_edges)

            probs = torch.softmax(p_bin, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(p_bin, dim=1).cpu().numpy()
            all_preds_bin.extend(preds)
            all_probs_bin.extend(probs)
            all_trues_bin.extend(batch_labels[:, 0].numpy())

            type_preds = torch.argmax(p_type, dim=1).cpu().numpy()
            all_preds_type.extend(type_preds)
            all_trues_type.extend(batch_labels[:, 1].numpy())

    y_true_bin = np.array(all_trues_bin)
    y_pred_bin = np.array(all_preds_bin)
    y_prob_bin = np.array(all_probs_bin)
    y_true_type = np.array(all_trues_type)
    y_pred_type = np.array(all_preds_type)

    print("1. BINARY CLASSIFICATION (Interaction: Yes/No)")
    print(f"{'-'*46}")

    acc = accuracy_score(y_true_bin, y_pred_bin)
    f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)

    try:
        roc_auc = roc_auc_score(y_true_bin, y_prob_bin)
    except:
        roc_auc = float('nan')

    print(f"• Accuracy:      {acc:.4f}  {'(✓)' if acc >= 0.90 else '(Target: 0.90)'}")
    print(f"• F1 Score:      {f1:.4f}  {'(✓)' if f1 >= 0.85 else ' (Target: 0.85)'}")
    print(f"• Precision:     {prec:.4f}")
    print(f"• Recall:        {rec:.4f}  {'(✓)' if rec >= 0.85 else ' (Important: catch interactions!)'}")
    print(f"• ROC-AUC:       {roc_auc:.4f}")

    cm = confusion_matrix(y_true_bin, y_pred_bin)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n1.2 CONFUSION MATRIX:")
    print(f"-"*61)
    print(f"| {' '*16}| Predicted: No Int | Predicted: Interact |")
    print(f"| Actual No Int   | {tn:6d}{' '*12}|{fp:6d} {' '*14}|")
    print(f"| Actual Interact | {fn:6d}{' '*12}|{tp:6d} {' '*14}|")
    print(f"-"*61)
    print(f"• True Positives:  {tp:6d}  (Correctly detected interactions)")
    print(f"• False Positives: {fp:6d}  (False alarms - Incorrectly predicted as safe)")
    print(f"• False Negatives: {fn:6d}  (⚠ CRITICAL: Missed interactions!)")
    print(f"• True Negatives:  {tn:6d}  (Correctly identified safe)")

    print("\n\n2. TYPE CLASSIFICATION (Which interaction type?)")
    print(f"{'-'*48}")

    pos_mask = y_true_bin == 1
    if pos_mask.sum() > 0:
        y_true_type_pos = y_true_type[pos_mask]
        y_pred_type_pos = y_pred_type[pos_mask]

        type_acc = accuracy_score(y_true_type_pos, y_pred_type_pos)
        type_f1 = f1_score(y_true_type_pos, y_pred_type_pos, average='weighted', zero_division=0)
        type_recall = recall_score(y_true_type_pos, y_pred_type_pos, average='weighted', zero_division=0)
        type_precision = precision_score(y_true_type_pos, y_pred_type_pos, average='weighted', zero_division=0)

        print(f"• 'Type' Accuracy: {type_acc:.4f}")
        print(f"• 'Type' F1 Score: {type_f1:.4f}")
        print(f"• 'Type' Precision:  {type_precision:.4f}")
        print(f"• 'Type' Recall:     {type_recall:.4f}")  
        print(f"• Total 'Types': {len(np.unique(y_true_type_pos))} unique types in test set")

        from collections import Counter
        type_counts = Counter(y_true_type_pos)
        print(f"\n• Top 5 Most Common Interaction Types:")
        for type_id, count in type_counts.most_common(5):
            type_name = graph['encoder'].inverse_transform([type_id])[0]
            accuracy = (y_pred_type_pos[y_true_type_pos == type_id] == type_id).mean()
            print(f"  - Type {type_name}: {count:4d} samples, {accuracy:.2%} accuracy")

    return {
        'binary': {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec, 'roc_auc': roc_auc},
        'type': {'accuracy': type_acc, 'f1': type_f1} if pos_mask.sum() > 0 else {},
        'confusion_matrix': {'true_positve': tp, 'false_positive': fp, 'true_negtive': tn, 'false_negative': fn},
    }

test_results = training_evaluation(trained_model, graph)


In [ ]:
def find_best_threshold(model, graph, device):
    model.eval()
    val_edges = graph['edge_index'][:, graph['val_mask']]
    val_labels_bin = graph['y_binary'][graph['val_mask']]

    val_loader = DataLoader(
        TensorDataset(val_edges.t(), val_labels_bin.unsqueeze(1)),
        batch_size=1024, shuffle=False
    )

    all_probs, all_trues = [], []
    with torch.no_grad():
        with autocast(device.type):
            node_emb = model.get_node_embeddings(
                graph['x_v1'].to(device), graph['x_v2'].to(device),
                graph['x_v3'].to(device), graph['edge_index'].to(device)
            )
        for batch_edges, batch_y in val_loader:
            p_bin, _ = model.forward_edges_from_emb(node_emb, batch_edges.t().to(device))
            probs = torch.softmax(p_bin, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_trues.extend(batch_y.squeeze().numpy())

    y_true = np.array(all_trues)
    y_prob = np.array(all_probs)

    print(f"\n{'—'*55}")
    print(f"{'Thresh':<8} | {'Acc':<8} | {'F1':<8} | {'Prec':<8} | {'Rec':<8}")
    print(f"{'—'*55}")

    best_acc, best_thresh = 0, 0.5
    for thresh in np.arange(0.3, 0.75, 0.05):
        preds = (y_prob >= thresh).astype(int)
        acc  = accuracy_score(y_true, preds)
        f1   = f1_score(y_true, preds, zero_division=0)
        prec = precision_score(y_true, preds, zero_division=0)
        rec  = recall_score(y_true, preds, zero_division=0)
        marker = " ← best acc" if acc > best_acc else ""
        print(f"  {thresh:.2f}   | {acc:.4f}  | {f1:.4f}  | {prec:.4f}  | {rec:.4f}{marker}")
        if acc > best_acc:
            best_acc = acc
            best_thresh = thresh

    print(f"{'—'*55}")
    print(f"Best threshold: {best_thresh:.2f} → Accuracy: {best_acc:.4f}")
    return best_thresh

BEST_THRESHOLD = find_best_threshold(model, graph, DEVICE)
print(f"Best threshold to use for predictions: {BEST_THRESHOLD:.2f}")


In [ ]:
current_log = {
    "training_data": TIMESTAMP,
    "total_training duration": training_history["training_duration"],
    "training_configuration": CONFIG,
    "final_training_metadata": {
        "training_loss": round(training_history["train_loss"][-1], 3),
        "validation_accuracy": round(training_history["val_acc"][-1], 3),
        "validation_f1-Score": round(training_history["val_f1"][-1], 3),
        "validation_precision": round(training_history["val_precision"][-1], 3),
        "validation_recall": round(training_history["val_recall"][-1], 3),
    },    
    "model_evaluation_result": {
        "binary_classification": {
            "accuracy":  round(test_results["binary"]["accuracy"],  3),
            "f1_Score":  round(test_results["binary"]["f1"],        3),
            "precision": round(test_results["binary"]["precision"], 3),
            "recall":    round(test_results["binary"]["recall"],    3),
            "roc_auc":   round(test_results["binary"]["roc_auc"],   3),
        },
        "Type Classification": {
            "accuracy": round(test_results["type"]["accuracy"], 3),
            "f1_score": round(test_results["type"]["f1"],       3),
        },
        "true_positve": int(test_results['confusion_matrix']['true_positve']),
        "false_positive": int(test_results['confusion_matrix']['false_positive']),
        "true_negtive": int(test_results['confusion_matrix']['true_negtive']),
        "false_negative": int(test_results['confusion_matrix']['false_negative']),
    },
    "best_threshold": f"{BEST_THRESHOLD:.3f}",
    "validation_accuracy_history": training_history["val_acc"],
    "training_loss_history": training_history["train_loss"],
}

log_file = f"{MODEL_METADATA_PATH}.json"
os.makedirs(os.path.dirname(log_file), exist_ok=True)

if os.path.exists(log_file):
    with open(log_file, "r") as f:
        try:
            logs_list = json.load(f)
            if not isinstance(logs_list, list):
                logs_list = []
        except json.JSONDecodeError:
            logs_list = []
else:
    logs_list = []

logs_list.append(current_log)

with open(log_file, "w") as f:
    json.dump(logs_list, f, indent=4)
print(f"(✓) Log successfully saved to {log_file}!\n")

inference_data = {
    'x_v1': graph['x_v1'].cpu(),
    'x_v2': graph['x_v2'].cpu(),
    'x_v3': graph['x_v3'].cpu(),
    'edge_index': graph['edge_index'].cpu(),
    'drug_map': graph['drug_map'],
    'n_types': graph['n_types'],
    'encoder': graph['encoder']
}
torch.save(inference_data, f'models/PharmGAT_v2_graph_data_{TIMESTAMP}.pt')
print(f"(✓) Inference data saved to: models/PharmGAT_v2_graph_data_{TIMESTAMP}.pt")


In [ ]:
from itertools import combinations

with open(DDI_TYPE_MAP, 'r') as f:
    ddi_type_map = json.load(f)

def predict_ddi_multi(model, graph, threshold=0.5):

    drugs_num = int(input("\nEnter total number of drugs (2 to 4): "))
    if drugs_num < 2 or drugs_num > 4:
        raise ValueError(f"Invalid number of drugs: {drugs_num}. Must be between 2 and 4.")

    drug_list = []
    print(f"\nPlease enter {drugs_num} UNIQUE drug IDs (e.g., DB00001):")

    for idx in range(drugs_num):
        drug_id = input(f"Drug {idx+1}: ").strip().upper()

        if not drug_id:
            raise ValueError("Drug ID cannot be empty.")

        if drug_id in drug_list:
            raise ValueError(f"Duplicate drug detected: '{drug_id}' already entered. Drugs must be unique.")

        if drug_id not in graph['drug_map']:
            raise ValueError(f"Drug ID '{drug_id}' not found in database.")

        drug_list.append(drug_id)

    print(f"\n✓ Received {len(drug_list)} unique drugs: {drug_list}")

    drug_pairs = list(combinations(drug_list, 2))

    model.eval()
    all_results = []

    for pair_idx, (drug_a_id, drug_b_id) in enumerate(drug_pairs, 1):
        print(f"\n[Pair {pair_idx}/{len(drug_pairs)}] Analyzing: {drug_a_id} + {drug_b_id}")
        print("-" * 60)

        u = graph['drug_map'][drug_a_id]
        v = graph['drug_map'][drug_b_id]

        query_node_indices = torch.tensor([u, v], dtype=torch.long)

        subset, edge_index, mapping, edge_mask = k_hop_subgraph(
            node_idx=query_node_indices, 
            num_hops=1, 
            edge_index=graph['edge_index'], 
            relabel_nodes=True
        )

        subgraph_u = mapping[0]
        subgraph_v = mapping[1]
        query_edge = torch.tensor([[subgraph_u], [subgraph_v]], device=DEVICE)

        x_v1 = graph['x_v1'][subset].to(DEVICE)
        x_v2 = graph['x_v2'][subset].to(DEVICE)
        x_v3 = graph['x_v3'][subset].to(DEVICE)
        edge_index = edge_index.to(DEVICE)

        with torch.no_grad():
            logits_bin, logits_type = model(x_v1, x_v2, x_v3, edge_index, query_edge)

            probs_bin = torch.softmax(logits_bin, dim=1)[0]
            prob_interaction = probs_bin[1].item()
            binary_pred = 1 if prob_interaction > threshold else 0

            probs_type = torch.softmax(logits_type, dim=1)[0]
            type_idx = torch.argmax(logits_type, dim=1).item()
            type_prob = probs_type[type_idx].item()

        interaction_effect = ddi_type_map.get(str(type_idx), "Unknown interaction type")

        if binary_pred == 1:
            print(f"INTERACTION DETECTED (Not Safe)")
            print(f"Interaction Probability:    {prob_interaction:.2%}")
            print(f"Interaction Type:           {type_idx}")
            print(f"Interaction Effect:         {interaction_effect}")
            print(f"Type Confidence:            {type_prob:.2%}")
        else:
            print(f"NO INTERACTION (Safe)")
            print(f"Safety Probability:         {(1-prob_interaction):.2%}")

        result = {
            'pair_number': pair_idx,
            'drug_a': drug_a_id,
            'drug_b': drug_b_id,
            'binary_prediction': binary_pred,
            'interaction_probability': prob_interaction,
            'interaction_type': type_idx,
            'interaction_effect': interaction_effect,
            'type_probability': type_prob,
            'safe': binary_pred == 0
        }
        all_results.append(result)

    interactions_found = sum(1 for r in all_results if r['binary_prediction'] == 1)
    safe_pairs = len(all_results) - interactions_found

    return {
        'drugs': drug_list,
        'total_pairs': len(all_results),
        'interactions_found': interactions_found,
        'safe_pairs': safe_pairs,
        'detailed_results': all_results
    }

results = predict_ddi_multi(trained_model, graph, threshold=BEST_THRESHOLD)
